<a href="https://colab.research.google.com/github/armandoordonez/AI-Engineering/blob/main/tutorial_langchain_langsmith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial práctico: LangChain + LangSmith

**LangChain** sirve para construir aplicaciones con LLMs (prompts, cadenas, herramientas, agentes). **LangSmith** sirve para observarlas, depurarlas y evaluarlas.

**Contenido**
1. Instalación
2. Claves y configuración de LangSmith
3. Primera llamada al modelo
4. Prompts y cadenas (LCEL)
5. Salida estructurada
6. Herramientas y agentes
7. RAG básico
8. `@traceable`: trazar tu propio código
9. Evaluación con datasets
10. Memoria conversacional

> Requiere: cuenta en [smith.langchain.com](https://smith.langchain.com) y una API key de Anthropic **u** OpenAI. Usa la API de LangChain 1.x.

> **Nota de esta versión:** se ejecuta en local (no en Colab) y usa como proveedor el catálogo de **NVIDIA** ([build.nvidia.com](https://build.nvidia.com)) con el modelo `mistralai/mistral-large-4-1t-a52b`, en lugar de Anthropic u OpenAI. Solo cambian las celdas de instalación, claves y modelo.

## 1. Instalación

In [1]:
%pip install -qU langchain langchain-nvidia-ai-endpoints langsmith langgraph python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Claves y configuración de LangSmith

Con `LANGSMITH_TRACING=true`, todo lo que ejecutes con LangChain queda registrado en LangSmith automáticamente.
Las claves se leen de un archivo `.env` para que no queden guardadas en el notebook.

In [2]:
import os
from dotenv import load_dotenv

# Versión local: las claves se leen del archivo .env en lugar de los secretos de Colab
load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tutorial-langchain"

for name in ("LANGSMITH_API_KEY", "NVIDIA_API_KEY"):
    print(name, "cargada" if os.environ.get(name) else "NO cargada")

LANGSMITH_API_KEY cargada
NVIDIA_API_KEY cargada


## 3. Primera llamada al modelo

In [3]:
import warnings
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# La librería aún no reconoce este modelo y avisa en cada llamada, aunque sí admite herramientas y salida estructurada
warnings.filterwarnings("ignore", category=UserWarning, module="langchain_nvidia_ai_endpoints")

MODELO = "mistralai/mistral-large-4-1t-a52b"
model = ChatNVIDIA(model=MODELO, temperature=0, max_completion_tokens=1024, timeout=300)

resp = model.invoke("Explica qué es un LLM en una frase.")
print(resp.content)
print(resp.usage_metadata)  # tokens usados

Un LLM (Large Language Model o modelo de lenguaje grande) es un sistema de inteligencia artificial entrenado con enormes cantidades de texto que le permite comprender, procesar y generar lenguaje humano de manera coherente y contextual.
{'input_tokens': 25, 'output_tokens': 421, 'total_tokens': 446}


👉 Abre LangSmith → proyecto **tutorial-langchain**. Ahí verás esta llamada con su input, output, tokens y latencia.

## 4. Prompts y cadenas (LCEL)
El operador `|` encadena componentes: **prompt → modelo → parser**.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un profesor experto en {tema}. Responde en español, en máximo 3 frases."),
    ("human", "{pregunta}"),
])

chain = prompt | model | StrOutputParser()
print(chain.invoke({"tema": "finanzas", "pregunta": "¿Qué es el interés compuesto?"}))

El interés compuesto es el mecanismo financiero mediante el cual los intereses generados en un período se suman al capital inicial, de modo que en el siguiente período se calculan sobre ese nuevo monto total. Esto provoca un crecimiento exponencial del dinero, ya que cada vez se obtienen rendimientos sobre una base mayor. Por ello, cuanto más tiempo se mantenga la inversión, mayor será el efecto multiplicador de las ganancias acumuladas.


In [5]:
# Varias entradas en paralelo
respuestas = chain.batch([
    {"tema": "física", "pregunta": "¿Qué es la entropía?"},
    {"tema": "historia", "pregunta": "¿Qué fue la Ilustración?"},
])
for r in respuestas:
    print("-", r, "\n")

- La entropía es una magnitud termodinámica que mide el grado de desorden de un sistema o la cantidad de energía que no puede transformarse en trabajo útil. Desde la mecánica estadística, representa el logaritmo del número de microestados posibles asociados a un macroestado dado, reflejando la probabilidad de dicho estado. Según la segunda ley de la termodinámica, en todo proceso espontáneo la entropía total del universo tiende a aumentar, lo que determina la dirección irreversible del tiempo. 

- La Ilustración fue un movimiento intelectual y cultural que surgió en Europa durante el siglo XVIII, caracterizado por el predominio de la razón humana sobre la tradición, la superstición y el dogma religioso. Sus pensadores promovieron valores como la libertad, la igualdad, la tolerancia y los derechos naturales, cuestionando el absolutismo monárquico y las estructuras de poder tradicionales. Este ideario sentó las bases filosóficas de las revoluciones liberales de finales del siglo XVIII, c

In [6]:
# Streaming token a token
for chunk in chain.stream({"tema": "IA", "pregunta": "¿Qué es RAG?"}):
    print(chunk, end="", flush=True)

RAG (

Retrieval-Augmented

 Generation o Gener

ación Aumentada por

 Recuperación) es

 una arquitectura que combina

... (18 outputs left out)

 un dominio o privado

, reduciendo significativamente

 las

 alucinaciones.

## 5. Salida estructurada
Pide al modelo un objeto Pydantic en lugar de texto libre.

In [7]:
from pydantic import BaseModel, Field

class Resumen(BaseModel):
    titulo: str
    puntos_clave: list[str] = Field(description="3 a 5 ideas principales")
    sentimiento: str = Field(description="positivo, negativo o neutro")

extractor = model.with_structured_output(Resumen)

texto = '''La empresa reportó ventas récord en el tercer trimestre, impulsadas por
el canal digital. Sin embargo, los costos logísticos subieron 12% y la gerencia
anunció un plan de eficiencia para 2027.'''

r = extractor.invoke(f"Resume este texto:\n{texto}")
r

Resumen(titulo='Resultados del Tercer Trimestre: Ventas Récord y Desafíos Logísticos', puntos_clave=['La empresa reportó ventas récord en el tercer trimestre impulsadas por el canal digital', 'Los costos logísticos aumentaron un 12% durante el período', 'La gerencia anunció un plan de eficiencia para implementar en 2027'], sentimiento='neutro')

## 6. Herramientas y agentes
El agente decide cuándo llamar cada herramienta.

In [8]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def clima(ciudad: str) -> str:
    """Devuelve el clima actual de una ciudad."""
    return f"En {ciudad} hay 24°C y está soleado."  # aquí llamarías una API real

@tool
def multiplicar(a: float, b: float) -> float:
    """Multiplica dos números."""
    return a * b

agent = create_agent(
    model=model,
    tools=[clima, multiplicar],
    system_prompt="Eres un asistente útil. Usa herramientas cuando las necesites. Responde en español.",
)

result = agent.invoke({"messages": [
    {"role": "user", "content": "¿Qué clima hace en Cali? Y ¿cuánto es 23.5 por 4?"}
]})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

¿Qué clima hace en Cali? Y ¿cuánto es 23.5 por 4?
================================== Ai Message ==================================
Tool Calls:
  clima (0141aa1f6)
 Call ID: 0141aa1f6
  Args:
    ciudad: Cali
  multiplicar (28617c7e7)
 Call ID: 28617c7e7
  Args:
    a: 23.5
    b: 4
================================= Tool Message =================================
Name: clima

En Cali hay 24°C y está soleado.
================================= Tool Message =================================
Name: multiplicar

94.0
================================== Ai Message ==================================

Aquí tienes las respuestas a tus dos preguntas:

🌤️ **Clima en Cali:** Hay **24°C** y está **soleado**.

🔢 **23.5 × 4 = 94**


👉 En LangSmith verás el árbol del agente: modelo → herramienta(s) → modelo. Esto ayuda mucho a depurar.

## 7. RAG básico
Responder con base en tus propios documentos. Para no depender de otra API key, aquí se usa un buscador **BM25** (por palabras clave).
En producción normalmente usarías embeddings y una base vectorial (Chroma, FAISS, pgvector...).

In [9]:
%pip install -qU langchain-community rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_community.retrievers import BM25Retriever

politicas = [
    "Vacaciones: cada empleado tiene 15 días hábiles de vacaciones al año.",
    "Trabajo remoto: se permite trabajo remoto 2 días por semana, previa aprobación del jefe.",
    "Reembolsos: los reembolsos de gastos se solicitan antes del día 25 de cada mes.",
    "Capacitación: la empresa financia hasta el 70% de cursos relacionados con el cargo.",
]
retriever = BM25Retriever.from_texts(politicas, k=2)

@tool
def buscar_politicas(consulta: str) -> str:
    """Busca en las políticas internas de la empresa."""
    return "\n".join(d.page_content for d in retriever.invoke(consulta))

rag_agent = create_agent(
    model,
    tools=[buscar_politicas],
    system_prompt="Responde SOLO con base en las políticas encontradas. Si no está, di que no sabes.",
)

out = rag_agent.invoke({"messages": [{"role": "user", "content": "¿Cuántos días de remoto puedo hacer?"}]})
print(out["messages"][-1].content)

/tmp/ipykernel_21724/1248073928.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Según las políticas internas de la empresa, **se permite trabajo remoto 2 días por semana**, previa aprobación del jefe.


**Con PDFs reales** (opcional):
```python
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
docs = PyPDFLoader("mi_documento.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(docs)
retriever = BM25Retriever.from_documents(chunks, k=4)
```

## 8. `@traceable`: trazar tu propio código
Así puedes rastrear en LangSmith funciones que no son de LangChain.

In [11]:
from langsmith import traceable

@traceable
def clasificar(texto: str) -> str:
    return "facturación" if "factura" in texto.lower() else "atención general"

@traceable(name="pipeline_soporte", tags=["tutorial"], metadata={"version": "v1"})
def responder_ticket(texto: str) -> str:
    categoria = clasificar(texto)
    return chain.invoke({"tema": categoria, "pregunta": texto})

print(responder_ticket("No me llegó la factura de septiembre, ¿qué hago?"))

Primero, revisa tu bandeja de spam o el portal de cliente del proveedor, ya que a veces los correos se filtran automáticamente. Si no la encuentras, contacta directamente con el departamento de facturación de la empresa emisora para solicitar que te reenvíen el documento o te proporcionen una copia. Es fundamental contar con la factura para tus registros contables y obligaciones fiscales, así que actúa cuanto antes.


## 9. Evaluación con datasets
1. Creas un dataset con entradas y salidas esperadas.
2. Defines la función a evaluar (*target*) y uno o más evaluadores.
3. Corres el experimento y lo comparas en la interfaz de LangSmith.

In [12]:
from langsmith import Client

client = Client()
DATASET = "preguntas-basicas-tutorial"

if not client.has_dataset(dataset_name=DATASET):
    ds = client.create_dataset(DATASET, description="Dataset de prueba del tutorial")
    client.create_examples(dataset_id=ds.id, examples=[
        {"inputs": {"pregunta": "¿Cuál es la capital de Colombia?"}, "outputs": {"respuesta": "Bogotá"}},
        {"inputs": {"pregunta": "¿Cuánto es 2 + 2?"},                "outputs": {"respuesta": "4"}},
        {"inputs": {"pregunta": "¿En qué continente está Egipto?"},  "outputs": {"respuesta": "África"}},
    ])

In [13]:
qa = ChatPromptTemplate.from_template("Responde solo con la respuesta, sin explicación: {pregunta}") | model | StrOutputParser()

def target(inputs: dict) -> dict:
    return {"respuesta": qa.invoke(inputs)}

# Evaluador 1: coincidencia simple
def contiene_respuesta(outputs: dict, reference_outputs: dict) -> bool:
    return reference_outputs["respuesta"].lower() in outputs["respuesta"].lower()

# Evaluador 2: brevedad
def es_breve(outputs: dict) -> bool:
    return len(outputs["respuesta"].split()) <= 5

resultados = client.evaluate(
    target,
    data=DATASET,
    evaluators=[contiene_respuesta, es_breve],
    experiment_prefix="v1",
)
resultados.to_pandas()

View the evaluation results for experiment: 'v1-1c709801' at:
https://smith.langchain.com/o/e6a7794d-3f40-49da-a64a-a6377215397f/datasets/d130b851-e92e-4934-988c-228d78ad9313/compare?selectedSessions=8990342e-090a-4821-b8f3-58a565a228ab




,inputs.pregunta,outputs.respuesta,error,reference.respuesta,feedback.contiene_respuesta,feedback.es_breve,execution_time,example_id,id
0,¿En qué continente está Egipto?,África,None,África,True,True,4.285549,2c0c0ae6-ef5e-444a-a318-7057eeadf2d1,01a1a842-0327-7484-a8ba-226e68e8519e
1,¿Cuál es la capital de Colombia?,Bogotá,None,Bogotá,True,True,2.404296,a0954381-ba01-4b98-a43c-37ce33a8452c,01a1a842-13e7-704f-b7ce-7084d2c2f511
2,¿Cuánto es 2 + 2?,4,None,4,True,True,1.663001,f522537a-2808-49f2-8d12-6143ae9e0ac4,01a1a842-1d4e-774c-9abb-0ba0ee228a2f


### LLM como juez (para respuestas abiertas)

In [14]:
class Veredicto(BaseModel):
    correcto: bool
    razon: str

juez = model.with_structured_output(Veredicto)

def llm_juez(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    v = juez.invoke(
        f"Pregunta: {inputs['pregunta']}\n"
        f"Respuesta de referencia: {reference_outputs['respuesta']}\n"
        f"Respuesta del modelo: {outputs['respuesta']}\n"
        "¿La respuesta del modelo es correcta respecto a la referencia?"
    )
    return {"key": "llm_juez", "score": v.correcto, "comment": v.razon}

client.evaluate(target, data=DATASET, evaluators=[llm_juez], experiment_prefix="v1-juez")

View the evaluation results for experiment: 'v1-juez-1f69d15b' at:
https://smith.langchain.com/o/e6a7794d-3f40-49da-a64a-a6377215397f/datasets/d130b851-e92e-4934-988c-228d78ad9313/compare?selectedSessions=6db42390-07a6-453c-b9f1-66d9ee3908ec




,inputs.pregunta,outputs.respuesta,error,reference.respuesta,feedback.llm_juez,execution_time,example_id,id
0,¿En qué continente está Egipto?,África,None,África,True,3.116402,2c0c0ae6-ef5e-444a-a318-7057eeadf2d1,01a1a842-2819-7f42-8b86-02b880b8774d
1,¿Cuál es la capital de Colombia?,Bogotá,None,Bogotá,True,6.315023,a0954381-ba01-4b98-a43c-37ce33a8452c,01a1a842-55be-7cbe-b741-1fdd3825cb80
2,¿Cuánto es 2 + 2?,4,None,4,True,2.054260,f522537a-2808-49f2-8d12-6143ae9e0ac4,01a1a842-99a0-71b0-a100-c01c5c8ad67f


👉 En LangSmith → **Datasets & Experiments** puedes comparar los experimentos lado a lado cuando cambies el prompt o el modelo.

## 10. Memoria conversacional
Con un *checkpointer*, el agente recuerda la conversación de cada `thread_id`.

In [15]:
from langgraph.checkpoint.memory import InMemorySaver

agente_memoria = create_agent(model, tools=[clima], checkpointer=InMemorySaver(),
                              system_prompt="Eres un asistente amable. Responde en español.")
config = {"configurable": {"thread_id": "jose-1"}}

agente_memoria.invoke({"messages": [{"role": "user", "content": "Hola, me llamo José y vivo en Cali."}]}, config)
r = agente_memoria.invoke({"messages": [{"role": "user", "content": "¿Cómo me llamo y qué clima hace donde vivo?"}]}, config)
print(r["messages"][-1].content)

¡Hola de nuevo, José! 😊

Te llamas **José** y vives en **Cali**, donde ahora mismo hay **24°C y está soleado** ☀️. ¡Qué buen clima para disfrutar el día!


## Próximos pasos
- **Prompt Hub**: versiona prompts con `client.push_prompt("mi-prompt", object=prompt)` y recupéralos con `client.pull_prompt("mi-prompt")`.
- **LangGraph**: para flujos con varios pasos, aprobación humana o varios agentes.
- **Producción**: monitoreo, alertas y *online evaluators* en LangSmith sobre el tráfico real.
- Documentación: [docs.langchain.com](https://docs.langchain.com)